## 01. Ridge Baseline Analysis

La ejecución del experimento se trasladó a `scripts/train.py`. Este notebook consulta los datos y resultados para análisis; no entrena modelos ni registra ejecuciones.

Desde la raíz del proyecto:
```text
python scripts/train.py
```

### Experiment Configuration

- Ridge(alpha=1.0, tol=0.0001, solver="auto", random_state=42).
- Sin Feature Selection ni transformaciones de distribución.
- Imputación numérica por mediana y StandardScaler.
- Encoding con los componentes existentes.
- SalePrice original.
- KFold de 5 folds, shuffle=True, random_state=42.
- MAE, RMSE y R² por fold, promedio y desviación estándar con ddof=0.

El holdout queda reservado. Las estadísticas se ajustan dentro de cada fold.

El script admite --alpha, --tol, --solver y --seed. Cada corrida registra CSV/JSON de métricas y el pipeline final ajustado con entrenamiento. Las versiones están documentadas en MLflow y en los artefactos del modelo; las dependencias del proyecto siguen en pyproject.toml y uv.lock.

In [ ]:
import pandas as pd
from IPython.display import display
from house_prices_mlops.pipeline import load_stage
from house_prices_mlops.dataset import get_predictors_and_target

df = load_stage("common")
X, y = get_predictors_and_target(df)
print("Training rows:", len(X))
print("Predictors before encoding:", X.shape[1])
display(X.dtypes.value_counts().rename("columns"))
display(y.describe())


### Review Recorded Results

La celda siguiente solo consulta la última ejecución finalizada del experimento Ridge. No ejecuta fit, validación cruzada ni registros en MLflow.

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri="http://mlflow:5000")
experiment = client.get_experiment_by_name("house-prices-ridge-experiments")
if experiment is None:
    raise ValueError("No existe el experimento; ejecute python scripts/train.py primero.")
runs = client.search_runs(
    [experiment.experiment_id],
    filter_string="attributes.run_name = 'ridge_baseline_standard_no_selection_no_distributions' AND attributes.status = 'FINISHED'",
    order_by=["attributes.start_time DESC"], max_results=1,
)
if not runs:
    raise ValueError("No hay ejecuciones Ridge finalizadas; ejecute scripts/train.py primero.")
run = runs[0]
display(pd.DataFrame([{key: run.data.params[key] for key in ("alpha", "tol", "solver", "seed")}]))
summary = pd.DataFrame({
    metric: {statistic: run.data.metrics[f"{metric}_{statistic}"] for statistic in ("mean", "std")}
    for metric in ("mae", "rmse", "r2")
})
display(summary)

fold_records = []
for metric in ("mae", "rmse", "r2"):
    for event in client.get_metric_history(run.info.run_id, f"fold_{metric}"):
        fold_records.append({"fold": event.step, "metric": metric, "value": event.value})
fold_metrics = pd.DataFrame(fold_records).pivot(index="fold", columns="metric", values="value")
display(fold_metrics)
print("MLflow run:", run.info.run_id)
print(f"http://localhost:5000/#/experiments/{experiment.experiment_id}/runs/{run.info.run_id}")


### Interpretation

MAE y RMSE se expresan en la escala original de SalePrice. La desviación estándar resume la variación entre folds, no un intervalo de confianza. Revise especialmente el fold con mayor RMSE antes de comparar esta referencia con nuevas alternativas. Los resultados de CV no sustituyen la evaluación final del holdout.